# Deception Classifier — Colab GPU Training
Run each cell in order. Make sure the runtime is set to **T4 GPU**:  
Runtime → Change runtime type → T4 GPU

## 1. Clone repo & install dependencies

In [ ]:
!git clone https://github.com/Lucca878/deceptionClassifierTraining.git
%cd deceptionClassifierTraining

In [2]:
!pip install -q -r requirements.txt

## 2. Verify GPU

In [3]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## 3. Train DistilBERT

In [4]:
!python src/pipeline/run_pipeline.py \
  --mode train \
  --model distilbert \
  --output_root ./outputs \
  --seed 42

## 4. Check output

In [ ]:
import os, glob

# Find the latest training run
run_dirs = sorted(glob.glob("outputs/distilbert_*"))
latest = run_dirs[-1] if run_dirs else None
print("Latest run:", latest)

if latest:
    for f in os.listdir(latest):
        print(" ", f)

In [ ]:
import pandas as pd

cv_path = os.path.join(latest, "cv_results.csv") if latest else None
if cv_path and os.path.exists(cv_path):
    df = pd.read_csv(cv_path)
    print(df)
    print("\nMean accuracy:", df["eval_accuracy"].mean().round(4))
    print("Mean F1:      ", df["eval_f1"].mean().round(4))

## 5. Download the trained model

In [ ]:
import shutil

model_dir = os.path.join(latest, "model") if latest else None
zip_path = "/content/distilbert_retrained"

if model_dir and os.path.exists(model_dir):
    shutil.make_archive(zip_path, "zip", model_dir)
    print("Zipped:", zip_path + ".zip")
else:
    print("Model dir not found:", model_dir)

In [ ]:
from google.colab import files
files.download("/content/distilbert_retrained.zip")